In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "420_build_gold_vw_dim_vendor.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.dim_vendor"
TARGET_VIEW = f"{catalog}.gold.vw_dim_vendor"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_dim_vendor
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    SELECT
          vendor_key                   AS `Vendor Key`
        , vendor_name                  AS `Vendor Name`
        , vendor_name_standardized     AS `Vendor Name Standardized`
        , project_count                AS `Project Count`
        , submission_count             AS `Submission Count`
        , low_bid_project_count        AS `Low Bid Project Count`
        , low_bid_submission_count     AS `Low Bid Submission Count`
        , latest_item_project_count    AS `Latest Item Project Count`
        , latest_item_row_count        AS `Latest Item Row Count`
        , latest_low_bid_project_count AS `Latest Low Bid Project Count`
        , first_actual_let_date        AS `First Actual Let Date`
        , last_actual_let_date         AS `Last Actual Let Date`
        , latest_first_actual_let_date AS `Latest First Actual Let Date`
        , latest_last_actual_let_date  AS `Latest Last Actual Let Date`
        , first_actual_let_year        AS `First Actual Let Year`
        , last_actual_let_year         AS `Last Actual Let Year`
        , source_row_count             AS `Source Row Count`
        , distinct_base_row_key_count  AS `Distinct Base Row Key Count`
        , distinct_source_row_id_count AS `Distinct Source Row ID Count`
        , latest_source_updated_at     AS `Latest Source Updated At`
        , latest_ingested_at_utc       AS `Latest Load Timestamp`
    FROM {SOURCE_TABLE}
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_VIEW}
    """).collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_dim_vendor successfully.")

    print("=" * 70)
    print("Built gold.vw_dim_vendor")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise